In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2,3"
print(os.environ["CUDA_VISIBLE_DEVICES"])

#### 项目概述
**基于大模型的文档检索问答**
本项目以大模型为中心构建一个问答系统，回答用户的汽车相关问题：先根据问题在文档中定位相关信息，再基于文档内容通过大模型生成答案。
#### 项目流程图
![框架图](pipeline.png)

In [ ]:
from retrievers.faiss_retriever import FaissRetriever
from vllm_model import Baichuan
from run import *

In [ ]:
data =  parse_pdf('data/train_a.pdf')


**改进1：采取更多路召回策略**
支持使用多个文本检索、向量检索，包括TF-IDF召回，m3e召回、bge召回、gte召回和bce召回，支持使用bge-reranker或bce-reranker-base_v1进行精排

In [ ]:
retrievers = []
for embed_model in ['m3e']:
    retrievers.append(FaissRetriever(embed_model, data))
print("faissretriever load ok")

In [ ]:
# BM25召回
retrievers.append(BM25(data))
print("bm25 load ok")

# TFIDF召回
retrievers.append(TFIDF(data))
print("tfidf load ok")

In [ ]:
# reRank模型
rerank = reRankLLM("pre_train_model/bce-reranker-base")
print("rerank model load ok")

**改进2：可配置基座大模型**
LLM分别采用ChatGLM3-6B, Qwen1.5-7B-Chat和Baichuan2-7B-Chat作为大模型基座，代码做成可配置。

In [ ]:
# LLM大模型
llm_dict = {
        "qwen": (ChatLLM, "pre_train_model/Qwen1.5-7B-Chat"),
        "baichuan": (ChatLLM, "pre_train_model/Baichuan2-7B-Chat"),
        "chatglm": (Baichuan, "pre_train_model/chatglm3-6b")
    }
llm_name = "qwen"
llm_model, llm_path = llm_dict[llm_name]
llm = llm_model(llm_path)
print(f"llm {llm_name} load ok")

利用prompt技术增强query来提升检索效果

In [ ]:
query = "如何通过中央显示屏进行副驾驶员座椅设置？"
queries = [query]

**改进3：检索前改写问题**
先用LLM先将问题改写和扩充一遍，然后将问题和这个改写后的问题拼接，提升检索效果。

In [ ]:
queries.append(llm.infer([rephrase_question_template(query)])[0])

**改进4: 先生成答案再检索**
先用LLM直接生成答案，然后将问题和这个生成的答案拼接，共同完成检索，提升检索效果。

In [ ]:
answer = llm.infer([answer_question_template(query)])[0]
if answer.strip() != "无答案":
    queries.append(query+" "+answer)

原始问题和通过上述两种方式扩展后的问题列表如下所示：

In [ ]:
for q in queries:
    print(q)
    print('------------------')

In [ ]:
batch_context = []
for q in queries:            
    docs = retrievers_recall(retrievers, q)
    batch_context.append(reRank(rerank, 6, q, docs))


In [ ]:
batch_qa_inputs = [qa_template(''.join(context), query) for context in batch_context]
batch_qa_outputs = llm.infer(batch_qa_inputs)

**改进5：先整理检索文档再回答**

将抽取后的文档使用LLM重新整理，使得杂乱知识库规整。然后再送入到答案生成模块

In [ ]:
rephrase_inputs = [rephrase_context_template(''.join(context)) for context in batch_context]
rephrase_context = llm.infer(rephrase_inputs)
batch_qa_inputs += [qa_template(''.join(context), query) for context in rephrase_context]

In [ ]:
rephrase_context

**改进6：递归生成答案**

一次给LLM一个检索到的文档，不断优化生成的答案，即利用prompt技术对LLM，context和原答案，优得到优化升级后的答案。

In [ ]:
for i in range(len(batch_context[0])):
    if i==0:
        recursive_qa_inputs = [qa_template(batch_context[0][i], query)]
    else:
        recursive_qa_inputs = [recursive_qa_template(batch_context[0][i], recursive_qa_outputs[0], query)]
    recursive_qa_outputs = llm.infer(recursive_qa_inputs)
    print(f"round {i}")
    print(recursive_qa_inputs[0])
    # print(recursive_qa_outputs[0])
batch_qa_outputs += recursive_qa_outputs

In [ ]:
for ans in batch_qa_outputs:
    print(ans)
    print('-----------------------------')